In [ ]:
import json
import re
import ast
import numpy as np
from collections import defaultdict

# =========================================================
# 1. ROBUST JSON EXTRACTION (handles ALL LLaMA mess)
# =========================================================

def extract_json(text):
    """
    Extracts last valid JSON block from noisy LLM output.
    Handles:
    - ```json fences
    - repeated JSON blocks
    - commentary text
    - partial truncation
    """

    if text is None:
        return None

    text = text.replace("```json", "").replace("```", "")

    # find all possible JSON objects
    candidates = re.findall(r"\{.*\}", text, re.DOTALL)

    for cand in reversed(candidates):
        try:
            return json.loads(cand)
        except:
            continue

    return None


# =========================================================
# 2. LOAD DATA
# =========================================================

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_gt_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    try:
        return json.loads(text)
    except:
        return ast.literal_eval(text)


# =========================================================
# 3. SCHEMA NORMALIZATION (CRITICAL FIX)
# =========================================================

def normalize_scenes(data):
    """
    Converts ANY schema into canonical form:
    {
        emotion, setting, atmosphere, state, theme
    }
    """

    scenes = []

    if isinstance(data, dict) and "output" in data:
        raw = data["output"]
    elif isinstance(data, dict) and "scenes" in data:
        raw = data["scenes"]
    elif isinstance(data, list):
        raw = data
    else:
        return []

    for i, s in enumerate(raw):

        scenes.append({
            "emotion": s.get("emotion"),
            "setting": s.get("setting"),
            "atmosphere": s.get("atmosphere"),
            "state": s.get("state"),
            "theme": s.get("theme"),
            "state_change": s.get("state_change")
        })

    return scenes


# =========================================================
# 4. METRIC UTILITIES
# =========================================================

def f1(gt_list, pred_list):
    gt_set = set(gt_list)
    pred_set = set(pred_list)

    tp = len(gt_set & pred_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return precision, recall, f1


def scene_count_score(gt, pred):
    return 1 - abs(len(gt) - len(pred)) / max(len(gt), 1)


# =========================================================
# 5. ATTRIBUTE F1
# =========================================================

def attr_f1(gt, pred, key):
    return f1(
        [s.get(key) for s in gt],
        [s.get(key) for s in pred]
    )[2]


# =========================================================
# 6. STATE TRANSITION SCORE
# =========================================================

def state_transition_f1(gt, pred):
    gt_trans = [s.get("state_change") for s in gt if s.get("state_change")]
    pred_trans = [s.get("state_change") for s in pred if s.get("state_change")]

    return f1(gt_trans, pred_trans)[2]


# =========================================================
# 7. CAUSAL PROXY SCORE
# =========================================================

def causal_proxy(gt, pred):
    gt_states = [s.get("state") for s in gt]
    pred_states = [s.get("state") for s in pred]

    match = 0
    n = min(len(gt_states), len(pred_states))

    for i in range(n):
        if gt_states[i] == pred_states[i]:
            match += 1

    return match / (n + 1e-9)


# =========================================================
# 8. FIXED SCHEMA COMPLIANCE (IMPORTANT FIX)
# =========================================================

def schema_compliance(pred):
    required = ["emotion", "setting", "atmosphere", "state", "theme"]

    total = 0
    correct = 0

    for s in pred:
        for f in required:
            total += 1
            if f in s and s[f] not in [None, ""]:
                correct += 1

    return correct / (total + 1e-9)


# =========================================================
# 9. MAIN EVALUATION LOOP
# =========================================================

def evaluate(gt_data, pred_data):

    results = defaultdict(list)

    for i, (gt_story, pred_story) in enumerate(zip(gt_data, pred_data)):

        pred_json = extract_json(pred_story["output"])

        if pred_json is None:
            continue

        gt_scenes = normalize_scenes(gt_story)
        pred_scenes = normalize_scenes(pred_json)

        # ---------------- metrics ----------------

        results["scene_acc"].append(scene_count_score(gt_scenes, pred_scenes))

        results["emotion_f1"].append(attr_f1(gt_scenes, pred_scenes, "emotion"))
        results["setting_f1"].append(attr_f1(gt_scenes, pred_scenes, "setting"))
        results["atmosphere_f1"].append(attr_f1(gt_scenes, pred_scenes, "atmosphere"))
        results["state_f1"].append(attr_f1(gt_scenes, pred_scenes, "state"))
        results["theme_f1"].append(attr_f1(gt_scenes, pred_scenes, "theme"))

        results["state_transition"].append(state_transition_f1(gt_scenes, pred_scenes))
        results["causal"].append(causal_proxy(gt_scenes, pred_scenes))
        results["schema"].append(schema_compliance(pred_scenes))

    # ---------------- final aggregation ----------------

    return {k: np.mean(v) for k, v in results.items()}


# =========================================================
# 10. RUN PIPELINE
# =========================================================

gt_path = "/content/Testing_Ground_Truth_Data.txt"
pred_path = "/content/all_story_predictions (1).json"

gt_data = load_gt_txt(gt_path)
pred_data = load_json(pred_path)

metrics = evaluate(gt_data, pred_data)

print("\n================ FINAL RESULTS ================\n")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


================ FINAL RESULTS ================

scene_acc: 0.7407
emotion_f1: 0.3228
setting_f1: 0.5000
atmosphere_f1: 0.8190
state_f1: 0.8294
theme_f1: 0.4407
state_transition: 0.3333
causal: 0.4815
schema: 0.8889


In [ ]:
import json
import re
import ast
import numpy as np
from collections import defaultdict

# =========================================================
# 1. LOAD DATA
# =========================================================

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_gt(path):
    with open(path, "r", encoding="utf-8") as f:
        text = f.read()
    try:
        return json.loads(text)
    except:
        return ast.literal_eval(text)

# =========================================================
# 2. ROBUST JSON EXTRACTION (LLAMA SAFE)
# =========================================================

def extract_json(text):
    if text is None:
        return None

    text = str(text)

    # remove markdown
    text = text.replace("```json", "").replace("```", "")

    # try direct parse
    try:
        return json.loads(text)
    except:
        pass

    # extract all JSON candidates
    candidates = re.findall(r"\{.*\}|\[.*\]", text, re.DOTALL)

    for cand in reversed(candidates):
        try:
            return json.loads(cand)
        except:
            continue

    return None

# =========================================================
# 3. NORMALIZE SCENES (CRITICAL FIX)
# =========================================================

def normalize(data):
    scenes = []

    if isinstance(data, dict):
        if "output" in data:
            data = data["output"]
        elif "scenes" in data:
            data = data["scenes"]
        elif "data" in data:
            data = data["data"]

    if isinstance(data, dict):
        data = [data]

    if not isinstance(data, list):
        return []

    for s in data:
        if isinstance(s, str):
            continue

        scenes.append({
            "emotion": s.get("emotion", "").strip().lower(),
            "setting": s.get("setting", "").strip().lower(),
            "atmosphere": s.get("atmosphere", "").strip().lower(),
            "state": s.get("state", "").strip().lower(),
            "theme": s.get("theme", "").strip().lower(),
            "state_change": s.get("state_change", "")
        })

    return scenes

# =========================================================
# 4. SAFE SET METRICS (PRECISION / RECALL / F1)
# =========================================================

def prf(gt_list, pred_list):
    gt_set = set([x for x in gt_list if x])
    pred_set = set([x for x in pred_list if x])

    tp = len(gt_set & pred_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)

    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)

    return precision, recall, f1

# =========================================================
# 5. ATTRIBUTE EVALUATION
# =========================================================

def attr_metrics(gt, pred, key):
    return prf(
        [s[key] for s in gt],
        [s[key] for s in pred]
    )

# =========================================================
# 6. STATE TRANSITION MATCH
# =========================================================

def state_transition(gt, pred):
    gt_t = [s.get("state_change", "").strip() for s in gt if s.get("state_change")]
    pr_t = [s.get("state_change", "").strip() for s in pred if s.get("state_change")]

    return prf(gt_t, pr_t)[2]

# =========================================================
# 7. CAUSAL CONSISTENCY (ORDER MATCH)
# =========================================================

def causal(gt, pred):
    gt_states = [s["state"] for s in gt]
    pr_states = [s["state"] for s in pred]

    n = min(len(gt_states), len(pr_states))
    if n == 0:
        return 0.0

    match = sum(gt_states[i] == pr_states[i] for i in range(n))
    return match / n

# =========================================================
# 8. SCHEMA VALIDITY
# =========================================================

def schema(pred):
    required = ["emotion", "setting", "atmosphere", "state", "theme"]

    total, valid = 0, 0

    for s in pred:
        for k in required:
            total += 1
            if k in s and s[k] not in ["", None]:
                valid += 1

    return valid / (total + 1e-9)

# =========================================================
# 9. SCENE ACCURACY
# =========================================================

def scene_acc(gt, pred):
    return 1 - abs(len(gt) - len(pred)) / max(len(gt), 1)

# =========================================================
# 10. MAIN EVALUATION LOOP
# =========================================================

def evaluate(gt_data, pred_data):

    results = defaultdict(list)

    for gt_story, pred_story in zip(gt_data, pred_data):

        pred_json = extract_json(pred_story.get("output"))

        if pred_json is None:
            continue

        gt_scenes = normalize(gt_story)
        pred_scenes = normalize(pred_json)

        if len(gt_scenes) == 0 or len(pred_scenes) == 0:
            continue

        # scene accuracy
        results["scene"].append(scene_acc(gt_scenes, pred_scenes))

        # attributes
        for key in ["emotion", "setting", "atmosphere", "state", "theme"]:
            p, r, f1 = attr_metrics(gt_scenes, pred_scenes, key)
            results[f"{key}_precision"].append(p)
            results[f"{key}_recall"].append(r)
            results[f"{key}_f1"].append(f1)

        # structure metrics
        results["state_transition"].append(state_transition(gt_scenes, pred_scenes))
        results["causal"].append(causal(gt_scenes, pred_scenes))
        results["schema"].append(schema(pred_scenes))

    return {k: float(np.mean(v)) for k, v in results.items() if len(v) > 0}

# =========================================================
# 11. RUN (CHANGE PATHS ONLY)
# =========================================================

gt_path = "/content/Testing_Ground_Truth_Data.txt"
pred_path = "/content/all_story_predictions (1).json"

gt_data = load_gt(gt_path)
pred_data = load_json(pred_path)

metrics = evaluate(gt_data, pred_data)

# =========================================================
# 12. PRINT FINAL RESULTS
# =========================================================

print("\n================ FINAL LLAMA 3 EVALUATION =================\n")

for k, v in sorted(metrics.items()):
    print(f"{k}: {v:.4f}")


================ FINAL LLAMA 3 EVALUATION =================

atmosphere_f1: 0.7714
atmosphere_precision: 0.7639
atmosphere_recall: 0.7963
causal: 0.4352
emotion_f1: 0.2143
emotion_precision: 0.2028
emotion_recall: 0.2407
scene: 0.7269
schema: 0.8889
setting_f1: 0.6574
setting_precision: 0.6296
setting_recall: 0.7222
state_f1: 0.7877
state_precision: 0.7324
state_recall: 0.8704
state_transition: 0.3791
theme_f1: 0.3315
theme_precision: 0.2361
theme_recall: 0.6667
